# 06 News Agent

Notebook-facing validation for the Tavily-backed LangChain news worker.

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from market_analyst.config.settings import load_settings
from market_analyst.telemetry import configure_notebook_logging
from market_analyst.services.agents.news import run_news_analysis_agent
from market_analyst.types.news import NewsAnalysisRequest

settings = load_settings()
logger = configure_notebook_logging(run_name="06_news_agent")
settings.require_chat_model()
settings.require_tavily()

c:\Users\rushi\OneDrive - ImmersiLearn Education Services LLP\Projects\LLM Projects\market-analyst-feb26\.venv\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer
2026-05-10 12:46:06,229 INFO 06_news_agent notebook_run_started


In [3]:
request = NewsAnalysisRequest(
    company_name="Reliance Industries Limited",
    ticker="RELIANCE.NS",
    sector="Energy, telecom, retail, and conglomerates in India",
    time_range="month",
    max_results=8,
)
request

NewsAnalysisRequest(company_name='Reliance Industries Limited', ticker='RELIANCE.NS', sector='Energy, telecom, retail, and conglomerates in India', question=None, time_range='month', max_results=8)

In [5]:
result = run_news_analysis_agent(settings, request)
result

2026-05-10 12:46:50,176 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"
2026-05-10 12:46:59,186 INFO httpx HTTP Request: POST https://ai-resources-e2e.cognitiveservices.azure.com/openai/deployments/gpt-5.4-mini/chat/completions?api-version=2025-04-01-preview "HTTP/1.1 200 OK"


NewsAnalysisResult(company_name='Reliance Industries Limited', ticker='RELIANCE.NS', sector='Energy, telecom, retail, and conglomerates in India', question='Find recent company-specific and sector-level news. Separate positive and negative developments, identify material stock implications, and assign a sentiment score from 0 to 100.', answer='{\n  "company_name": "Reliance Industries Limited",\n  "ticker": "RELIANCE.NS",\n  "sector": "Energy, telecom, retail, and conglomerates in India",\n  "sentiment_score": 46,\n  "positive_developments": [\n    "Reliance Retail continued to expand store count, with recent reporting indicating 820 net store additions in FY26, suggesting improving demand and continued execution in the retail business.",\n    "Reuters noted Reliance has balance-sheet buffers from its telecom and retail businesses, which can help offset volatility in the refining and oil-to-chemicals segment.",\n    "Market commentary suggests some analysts expect a recovery in refinin

In [6]:
print(result.answer)
print("sentiment_score=", result.sentiment_score)

{
  "company_name": "Reliance Industries Limited",
  "ticker": "RELIANCE.NS",
  "sector": "Energy, telecom, retail, and conglomerates in India",
  "sentiment_score": 46,
  "positive_developments": [
    "Reliance Retail continued to expand store count, with recent reporting indicating 820 net store additions in FY26, suggesting improving demand and continued execution in the retail business.",
    "Reuters noted Reliance has balance-sheet buffers from its telecom and retail businesses, which can help offset volatility in the refining and oil-to-chemicals segment.",
    "Market commentary suggests some analysts expect a recovery in refining conditions after the recent Middle East supply shock normalizes."
  ],
  "negative_developments": [
    "Reliance missed analyst estimates in its latest quarterly results, with profit falling 12.5% year on year.",
    "Core earnings in the refining business fell 3.7% year on year in the quarter, reflecting pressure from higher crude and input costs t

In [7]:
assert result.company_name
assert result.ticker
assert result.answer
assert result.sentiment_score is None or 0 <= result.sentiment_score <= 100